In [1]:
import pandas as pd
import numpy as np
from datasets import load_dataset
import tiktoken
import json
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import re
from groq import Groq
import os
from dotenv import load_dotenv
import random

load_dotenv()  # Load environment variables from a .env file if present
client = Groq(api_key=os.getenv("groq_api_key"))

d:\Work\Github\google-tunix-kaggle\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [23]:
math =pd.read_parquet("../data/math-distillation/math_distillation_dataset.parquet")
code =pd.read_parquet("../data/code-distillation/code_distillation_dataset.parquet")
science =pd.read_parquet("../data/science/science_distillation_dataset.parquet")
common_sense =pd.read_parquet("../data/common-sense/commonsense_distillation_dataset.parquet")
creative_writing =pd.read_parquet("../data/creative-writing/creative_writing_distillation_dataset.parquet")
ideation =pd.read_parquet("../data/ideation/ideation_distillation_dataset.parquet")
financial = pd.read_parquet("../data/financial/financial_distillation_dataset.parquet")
reading = pd.read_parquet("../data/reading/reading_distillation_dataset.parquet")
summarization = pd.read_parquet("../data/summarization/summarization_distillation_dataset.parquet")
#Function that adds a column called split if it's not there and sets it to 'train'
def ensure_split_column(df):
    if 'split' not in df.columns:
        df['split'] = 'train'
    return df
math = ensure_split_column(math)
code = ensure_split_column(code)
science = ensure_split_column(science)
common_sense = ensure_split_column(common_sense)
creative_writing = ensure_split_column(creative_writing)
ideation = ensure_split_column(ideation)
financial = ensure_split_column(financial)
reading = ensure_split_column(reading)
summarization = ensure_split_column(summarization)


combined_df = pd.concat([math, code, science, common_sense, creative_writing, ideation, financial, reading, summarization], ignore_index=True)


In [24]:
ideation[ideation['total_tokens'] < 2048]

,input,category,domain,ground_truth,source_answer,source,problem_type,question_type,split,uid,reasoning,response,input_tokens,output_tokens,total_tokens,success,error
0,Brainstorm 5 innovative solutions to reducing ...,problem_solving,creative_ideation,None,None,synthetic_ideation_prompts,brainstorming,open_ended,train,ideation1,The user asks to brainstorm 5 innovative solut...,"**5 High‑Impact, Feasible Ways Restaurants Can...",136,1328,1464,True,None
1,Brainstorm 5 innovative solutions to combating...,problem_solving,creative_ideation,None,None,synthetic_ideation_prompts,brainstorming,open_ended,train,ideation2,"We need to produce 5 innovative solutions, foc...","**5 Feasible, High‑Impact Solutions for Combat...",134,1507,1641,True,None
2,How might different industries approach solvin...,problem_solving,creative_ideation,None,None,synthetic_ideation_prompts,brainstorming,open_ended,train,ideation3,"The user asks: ""How might different industries...",**Three Industry‑Level Playbooks for Lightenin...,133,1393,1526,True,None
3,Brainstorm 5 innovative solutions to reducing ...,problem_solving,creative_ideation,None,None,synthetic_ideation_prompts,brainstorming,open_ended,train,ideation4,We need to brainstorm 5 innovative solutions t...,"**Five Feasible, High‑Impact Innovations to Li...",135,1637,1772,True,None
4,Brainstorm 5 innovative solutions to improving...,problem_solving,creative_ideation,None,None,synthetic_ideation_prompts,brainstorming,open_ended,train,ideation5,"We need to give 5 innovative solutions, feasib...","**5 Feasible, High‑Impact Solutions for Boosti...",134,1291,1425,True,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,Challenge assumptions: What if the problem of ...,unconventional_thinking,creative_ideation,None,None,synthetic_ideation_prompts,brainstorming,open_ended,train,ideation1996,"User asks: ""Challenge assumptions: What if the...",### Turning “Cost” from a Constraint into a Co...,131,1459,1590,True,None
1996,Challenge assumptions: What if the problem of ...,unconventional_thinking,creative_ideation,None,None,synthetic_ideation_prompts,brainstorming,open_ended,train,ideation1997,"The user asks: ""Challenge assumptions: What if...",### Rethinking Complexity: From Burden to Busi...,131,1331,1462,True,None
1997,Apply agile methodology from music to solve ch...,unconventional_thinking,creative_ideation,None,None,synthetic_ideation_prompts,brainstorming,open_ended,train,ideation1998,"We need to answer as an innovation consultant,...",### “Edu‑Jam” – An Agile‑in‑Music Playbook for...,128,1759,1887,True,None
1998,What's the opposite of conventional wisdom abo...,unconventional_thinking,creative_ideation,None,None,synthetic_ideation_prompts,brainstorming,open_ended,train,ideation1999,"We need to answer: ""What's the opposite of con...",**The “Anti‑Wellness” Lens – What Happens When...,130,1814,1944,True,None


In [25]:
##Filter to train data samples and those with reasoning and response populated
combined_df1 = combined_df[(combined_df['split'] == 'train') & 
                          (combined_df['reasoning'].notnull()) & 
                          (combined_df['response'].notnull())]

print(f"Filtered dataset size: {len(combined_df1)}")

#Filter to data with total_tokens < 1900
combined_df2 = combined_df1[combined_df1['total_tokens'] < 1900]
print(f"Filtered dataset size after token limit: {len(combined_df2)}")

Filtered dataset size: 45233
Filtered dataset size after token limit: 40788


In [4]:
combined_df2.sample(10)

,input,source_answer,split,source,domain,ground_truth,problem_type,question_type,uid,reasoning,...,error,model_answer,model_answer_available,validated_correct,difficulty,input_token_count,extracted_answer,is_correct,story_token_count,category
23639,Question: Which statement describes how muscle...,C,train,ai2_arc/ARC-Challenge,science,C,multiple_choice,multiple_choice,science844,We need to answer: Which statement describes h...,...,None,NaN,NaN,NaN,NaN,75.0,NaN,NaN,NaN,NaN
8817,The value of $(-2)^{3}-(-3)^{2}$ is\n(A) -17\n...,"Evaluating, $(-2)^{3}-(-3)^{2}=-8-9=-17$.\n\nA...",train,OpenR1-Math-220k,math,-17,Algebra,MCQ,math8818,We need compute (-2)^3 - (-3)^2.\n\n(-2)^3 = -...,...,None,None,False,False,NaN,NaN,NaN,NaN,NaN,NaN
19664,Create a MySQL query to group a table of stude...,"SELECT country, COUNT(*) FROM student_records ...",train,sahil2801/CodeAlpaca-20k,code,,code generation,code generation by instruction,code35065,"The user asks: ""Create a MySQL query to group ...",...,None,NaN,NaN,NaN,UNKNOWN_DIFFICULTY,NaN,NaN,NaN,NaN,NaN
5446,A building has 10 floors. It takes 15 seconds ...,"So for all even-numbered floors (2, 4, 6, 8, 1...",train,gsm8k,math,2,,,math5447,We need to interpret problem: building has 10 ...,...,None,2\text{ minutes,True,False,NaN,NaN,NaN,NaN,NaN,NaN
38126,Continue this story with a concluding sentence...,I was happy to get my wallet back.,train,igormorgado/ROCStories2018,creative_writing,I was happy to get my wallet back.,story_completion,story_completion,writing3475,We need to continue the story with a concludin...,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.0,NaN
43070,federal realty investment trust schedule iii s...,29441.0,train,FinGPT/fingpt-convfinqa,financial_reasoning,29441.0,financial_qa,Conversational financial QA,dquant2573,We need to parse the data. The user gave a tab...,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35531,Write a creative story based on this prompt:\n...,I pulled my winter cap down low over my ears. ...,train,euclaise/writingprompts,creative_writing,I pulled my winter cap down low over my ears. ...,generation,story_writing,writing880,We need to produce a creative story 300-500 wo...,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,142.0,NaN
6057,A tree on a farm has 10 branches. Each branch ...,If the tree's 10 branches have 40 sub-branches...,train,gsm8k,math,96000,,,math6058,We need to parse the problem: A tree has 10 br...,...,None,"96{,",True,False,NaN,NaN,NaN,NaN,NaN,NaN
24189,Question: High levels of radiation can remove ...,A,train,sciq,science,A,multiple_choice,multiple_choice,science1394,"The question: ""High levels of radiation can re...",...,None,NaN,NaN,NaN,NaN,29.0,NaN,NaN,NaN,NaN
24451,Question: What device measures current that fl...,A,train,sciq,science,A,multiple_choice,multiple_choice,science1656,"We need to answer: ""What device measures curre...",...,None,NaN,NaN,NaN,NaN,34.0,NaN,NaN,NaN,NaN


In [26]:
print(f"Total combined dataset size: {len(combined_df2)}")
print("Dataset sizes by domain:")
for domain in combined_df2['domain'].unique():
    domain_size = len(combined_df2[combined_df2['domain'] == domain])
    print(f"  {domain}: {domain_size}")
    
print("Dataset sizes by source:")
for source in combined_df2['source'].unique():
    source_size = len(combined_df2[combined_df2['source'] == source])
    print(f"  {source}: {source_size}")

Total combined dataset size: 40788
Dataset sizes by domain:
  math: 10651
  code: 6475
  science: 4380
  commonsense_reasoning: 6691
  creative_writing: 3828
  creative_ideation: 1818
  numerical_reasoning: 1998
  financial_reasoning: 1770
  reading_comprehension: 997
  summarization: 2180
Dataset sizes by source:
  gsm8k: 7464
  OpenR1-Math-220k: 2483
  Math/AIME 2025: 5
  ChilleD/SVAMP: 699
  google-research-datasets/mbpp: 338
  nvidia/OpenCodeReasoning: 120
  codeparrot/apps: 122
  sahil2801/CodeAlpaca-20k: 2990
  iamtarun/python_code_instructions_18k_alpaca: 2905
  ai2_arc/ARC-Challenge: 1119
  sciq: 3000
  Idavidrein/gpqa: 261
  tau/commonsense_qa: 2000
  tasksource/strategy-qa: 2290
  hotpot_qa: 2401
  euclaise/writingprompts: 1832
  igormorgado/ROCStories2018: 1996
  synthetic_ideation_prompts: 1818
  ibm/tatqa: 1998
  FinGPT/fingpt-convfinqa: 1770
  ehovy/race: 500
  FabianWillner/triviaQARC: 497
  abisee/cnn_dailymail: 2180


In [ ]:
##Words/sentences to clean or remove
remove_these=['We need to follow developer instructions.']

##Train - Validation - Test Split

In [28]:
from sklearn.model_selection import train_test_split

# Samples filtered out because split != 'train'
non_train_split = combined_df[(combined_df['split'] != 'train') & 
                               (combined_df['reasoning'].notnull()) & 
                               (combined_df['response'].notnull())]
print(f"Samples from non-train splits: {len(non_train_split)}")

# Samples that passed initial filters but exceeded token limit
dropped_samples = combined_df1[combined_df1['total_tokens'] >= 1900]
print(f"Samples dropped due to token limit: {len(dropped_samples)}")

# Your filtered training candidates
combined_df2 = combined_df1[combined_df1['total_tokens'] < 1900]
print(f"Samples for train/valid/test split: {len(combined_df2)}")

# First split: 90% train, 10% temp (which will become 5% valid, 5% test)
train_df, temp_df = train_test_split(
    combined_df2, 
    train_size=0.90, 
    stratify=combined_df2['domain'],
    random_state=42
)

# Second split: split the 10% temp into 50-50 (5% valid, 5% test of original)
valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df['domain'],
    random_state=42
)

# Add dropped samples and non-train splits to test set
test_df = pd.concat([test_df, dropped_samples, non_train_split], ignore_index=True)

print(f"\nFinal split:")
print(f"Train: {len(train_df)} ({len(train_df)/len(combined_df2)*100:.1f}%)")
print(f"Valid: {len(valid_df)} ({len(valid_df)/len(combined_df2)*100:.1f}%)")
print(f"Test: {len(test_df)} samples (base: {len(temp_df)//2}, +{len(dropped_samples)} long-context, +{len(non_train_split)} non-train splits)")

Samples from non-train splits: 600
Samples dropped due to token limit: 4445
Samples for train/valid/test split: 40788

Final split:
Train: 36709 (90.0%)
Valid: 2039 (5.0%)
Test: 7085 samples (base: 2039, +4445 long-context, +600 non-train splits)


In [29]:
train_df.to_parquet('../data//final_data//custom_df_train_sft.parquet')
valid_df.to_parquet('../data//final_data//custom_df_valid_sft.parquet')
test_df.to_parquet('../data//final_data//custom_df_test_sft.parquet')

In [30]:
train_df.sample(5)

,input,source_answer,split,source,domain,ground_truth,problem_type,question_type,uid,reasoning,...,error,model_answer,model_answer_available,validated_correct,difficulty,input_token_count,extracted_answer,is_correct,story_token_count,category
47054,Summarize the following news article:\n\n(CNN)...,Mas Selamat Kastari was arrested April 1 in Jo...,train,abisee/cnn_dailymail,summarization,Mas Selamat Kastari was arrested April 1 in Jo...,news_summary,summarization,summarization1783,We need to produce concise summary 100-200 wor...,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46632,Summarize the following news article:\n\n(CNN)...,"Marshal's body found in border town of Juarez,...",train,abisee/cnn_dailymail,summarization,"Marshal's body found in border town of Juarez,...",news_summary,summarization,summarization1361,We need to produce a concise summary 100-200 w...,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18494,"Reverse this sentence: ""I can do coding.""\n","""coding do can I""",train,sahil2801/CodeAlpaca-20k,code,,code generation,code generation by instruction,code27440,"The user asks to reverse the sentence: ""I can ...",...,None,NaN,NaN,NaN,UNKNOWN_DIFFICULTY,NaN,NaN,NaN,NaN,NaN
25775,Question: What type of reproduction is exempli...,A,train,sciq,science,A,multiple_choice,multiple_choice,science2980,"The user asks: ""What type of reproduction is e...",...,None,NaN,NaN,NaN,NaN,39.0,NaN,NaN,NaN,NaN
44223,shareowner return performance graph the follow...,0.9809,train,FinGPT/fingpt-convfinqa,financial_reasoning,0.9809,financial_qa,Conversational financial QA,dquant3726,We need to understand the question. They gave ...,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
os.getcwd()

'd:\\Work\\Github\\google-tunix-kaggle\\data-generation'

In [8]:
valid_df = pd.read_parquet("../data//final_data//custom_df_valid_sft.parquet")
# Define your target counts per domain
domain_counts = {
    'code': 10,
    'math': 10,
    'commonsense_reasoning': 20,
    'creative_ideation': 20,
    'creative_writing': 20,
    'financial_reasoning': 10,
    'numerical_reasoning': 10,
    'reading_comprehension': 20,
    'science': 20,
    'summarization': 20
}

# Sample each domain separately and concatenate
sampled_dfs = []
for domain, count in domain_counts.items():
    domain_samples = valid_df[valid_df['domain'] == domain].sample(
        n=min(count, len(valid_df[valid_df['domain'] == domain])),
        random_state=42  # for reproducibility
    )
    sampled_dfs.append(domain_samples)

valid_df_eval_sample = pd.concat(sampled_dfs, ignore_index=True)
valid_df_eval_sample.groupby('domain').size()

domain
code                     10
commonsense_reasoning    20
creative_ideation        20
creative_writing         20
financial_reasoning      10
math                     10
numerical_reasoning      10
reading_comprehension    20
science                  20
summarization            20
dtype: int64

In [11]:
valid_df_eval_sample=valid_df_eval_sample[['uid','input','ground_truth','domain','problem_type','reasoning','response']]
valid_df_eval_sample.to_parquet("../data//final_data//model_evaluation_sample.parquet")

In [13]:
print(valid_df_eval_sample.groupby('domain').size())

domain
code                     10
commonsense_reasoning    20
creative_ideation        20
creative_writing         20
financial_reasoning      10
math                     10
numerical_reasoning      10
reading_comprehension    20
science                  20
summarization            20
dtype: int64


In [16]:
valid_df_eval_sample.sample(10)

,uid,input,ground_truth,domain,problem_type,reasoning,response
95,dquant1373,"Table :\n"""","""","""",""Total Number of"",""Approxima...",(29.1-58.0)/58.0,numerical_reasoning,calculation,We need to extract numbers: March total number...,**Relevant numbers from the table**\n\n| Month...
30,commonsense1033,"The forgotten leftovers had gotten quite old, ...",B,commonsense_reasoning,multiple_choice,"We need to answer the question: ""The forgotten...",**B. refrigerator**
0,code18321,How can we check whether a given number is pri...,,code,code generation,"We need to answer: explain solution approach, ...",**Solution Approach**\n\n1. **Handle edge case...
14,math4544,Sarah interviewed 450 students at her school a...,113,math,,We need to find number who liked strawberries....,"Since every student chose exactly one fruit, t..."
133,science1890,"Question: Starch and glycogen, as well as simp...",A,science,multiple_choice,"The user asks: ""Starch and glycogen, as well a...","Starch, glycogen, glucose, and fructose are al..."
4,code25223,What is the output of the following Java code?...,,code,code generation,We need to answer: What is output? The code de...,**Explanation**\n\n- `x % y` is the **modulo (...
18,math6316,Jack is mad at his neighbors for blasting Tayl...,1450,math,,We need to compute damages: 3 tires at $250 ea...,Jack’s damages consist of:\n\n- **Tires:** 3 ×...
116,reading329,"Context: Wanted, Someone for a Kiss\nWe're loo...",D,reading_comprehension,reading_multiple_choice,We need to answer based on the ad for Father C...,**Answer: D. can work for about a month.**
135,science2029,Question: What measurement is used to describe...,A,science,multiple_choice,"We need to answer the question: ""What measurem...",The amount of water vapor present in the atmos...
57,ideation757,Combine wine corks with sensors to create some...,None,creative_ideation,brainstorming,User asks to combine wine corks with sensors t...,"**Product Concept: “CorkSense™ – Smart, Self‑P..."
